In [1]:
# import libraries
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# download historical data for SP500 from Yahoo Finance
ticker = '^GSPC'
period = '5y'
data = yf.download(ticker, period = period, multi_level_index = False)

# drop unnecessary columns and calculate returns
data['Returns'] = data['Close'].pct_change()
data = data[['Close', 'Returns']].dropna()

# create autoregressive features (lag = 5 design matrix)
lags = 5
X = pd.DataFrame(index=data.index)

for lag in range(1, lags + 1):
    X[f'Lag_{lag}'] = data['Returns'].shift(lag)

X = X.dropna()
X['intercept'] = 1  # add intercept term

# target variable
y = data['Returns'].loc[X.index]

[*********************100%***********************]  1 of 1 completed


,Lag_1,Lag_2,Lag_3,Lag_4,Lag_5,intercept
Date,,,,,,
2021-03-23,0.007025,-0.000603,-0.014761,0.002879,-0.001570,1
2021-03-24,-0.007631,0.007025,-0.000603,-0.014761,0.002879,1
2021-03-25,-0.005467,-0.007631,0.007025,-0.000603,-0.014761,1
2021-03-26,0.005240,-0.005467,-0.007631,0.007025,-0.000603,1
2021-03-29,0.016631,0.005240,-0.005467,-0.007631,0.007025,1
...,...,...,...,...,...,...
2026-03-06,-0.005647,0.007756,-0.009444,0.000398,-0.004339,1
2026-03-09,-0.013277,-0.005647,0.007756,-0.009444,0.000398,1
2026-03-10,0.008304,-0.013277,-0.005647,0.007756,-0.009444,1


In [ ]:
# compute OLS estimates using matrix algebra
beta = np.linalg.inv(X.T @ X) @ X.T @ y
beta = np.array(beta) # convert to numpy array for easier comparison

# coefficients using sklearn for comparison
model = LinearRegression(fit_intercept=False)
model.fit(X, y)
sklearn_beta = model.coef_
sklearn_beta

# print coefficients and check for errors
for i in range(len(beta)-1):
    print(f'OLS inverse beta_{i} = {np.round(beta[i], 4)}')

print(f'OLS inverse intercept = {np.round(beta[-1], 4)}')

print('-------')
for i in range(len(sklearn_beta)-1):
    print(f'Sklearn beta_{i} = {np.round(sklearn_beta[i], 4)}')

print(f'Sklearn intercept = {np.round(sklearn_beta[-1], 4)}')

print('-------')
print("Difference between OLS and sklearn coefficients:")
print(np.round(sklearn_beta - beta, 2 ))

OLS inverse beta_0 = -0.0247
OLS inverse beta_1 = -0.0061
OLS inverse beta_2 = -0.0699
OLS inverse beta_3 = -0.0417
OLS inverse beta_4 = -0.005
OLS inverse intercept = 0.0005
-------
Sklearn beta_0 = -0.0247
Sklearn beta_1 = -0.0061
Sklearn beta_2 = -0.0699
Sklearn beta_3 = -0.0417
Sklearn beta_4 = -0.005
Sklearn intercept = 0.0005
-------
Difference between OLS and sklearn coefficients:
[-0. -0.  0. -0.  0. -0.]
